# Figure: feature embedding geometry

Paper section: *Structural Properties of DIWECT Embeddings*, paragraph *Feature Embedding Geometry*


Uses EAS2D generated on the fly

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.manifold import MDS

import common as C

C.use_paper_style()
FOLDER = "3-feature-geometry"

SIZE, AREA = 64, 380
NUM_DIRS, NUM_HEIGHTS = 64, 48
N_PER_CLASS, N_SEEDS = 60, 5

METHODS = [("pixels", "baseline"), ("wect", "baseline"), ("wect_pca", "dim control"),
           ("diwect", "invariant"), ("wect_align", "oracle")]
ORDER = [m for m, _ in METHODS]
ROLE = dict(METHODS)
TICKS = {"pixels": "Pixels", "wect": "WECT", "wect_pca": "WECT\nPCA", "diwect": "DIWECT",
         "wect_align": "WECT\n+ align"}

shapes = C.make_shape_library(size=SIZE, area=AREA)
transform = C.WECTTransform(num_dirs=NUM_DIRS, num_heights=NUM_HEIGHTS)
K = len(shapes)

## Dataset and representations

`wect_pca` is the WECT projected by PCA to the DIWECT's width (48) as a dimension control. `wect_align` is the WECT of each image rotated back by its true angle (by an oracle).

In [ ]:
def make_dataset(seed):
    rng = np.random.default_rng(seed)
    images, labels, angles = [], [], []
    for label, img in enumerate(shapes.values()):
        for _ in range(N_PER_CLASS):
            a = rng.uniform(0.0, 360.0)
            images.append(C.rotate_image(img, a))
            labels.append(label)
            angles.append(a)
    return images, np.asarray(labels), np.asarray(angles)


def build(images, angles):
    W = np.stack([transform.wect(im) for im in images])
    flat = W.reshape(len(W), -1)
    aligned = np.stack([transform.wect(C.rotate_image(im, -a)) for im, a in zip(images, angles)])
    return {
        "pixels": np.stack([im.ravel() for im in images]),
        "wect": flat,
        "wect_pca": PCA(n_components=NUM_HEIGHTS).fit_transform(flat),
        "diwect": np.stack([transform.diwect(im) for im in images]),   # exact (src/diwect.py)
        "wect_align": aligned.reshape(len(W), -1),
    }


def within_group_ratio(d2, groups):
    """Mean within-group squared distance over the mean overall one (off-diagonal pairs)."""
    off = ~np.eye(len(d2), dtype=bool)
    same = (groups[:, None] == groups[None, :]) & off
    return float(d2[same].mean() / d2[off].mean())

## Within-class distances, over 5 seeds

In [ ]:
rows = []
for seed in range(N_SEEDS):
    images, y, angles = make_dataset(seed)
    pose = np.minimum((angles / (360.0 / K)).astype(int), K - 1)   # K rotation bins
    feats = build(images, angles)
    d2 = {name: cdist(feats[name], feats[name], "sqeuclidean") for name in ORDER}
    rows += [{"method": name, "seed": seed, "within_class": within_group_ratio(d2[name], y)} for name in ORDER]
    print(f"  seed {seed} done", flush=True)
last = {"d2": d2, "y": y, "pose": pose}   # the MDS panel uses the last seed

df = pd.DataFrame(rows)
agg = df.groupby("method").within_class.agg(["mean", "std"]).reindex(ORDER)
df.to_csv(C.figure_dir(FOLDER) / "figure3a-within-class.csv", index=False)   # per-seed values behind panel (a)
print(agg.round(3).to_string())

## MDS embeddings of the last seed

Metric MDS on the distance matrices themselves (not PCA on the features) so we visually represent the metric

In [ ]:
embeddings = {name: MDS(n_components=2, metric="precomputed", init="random", random_state=0,
                        normalized_stress="auto").fit_transform(np.sqrt(last["d2"][name]))
              for name in ("wect", "diwect")}

## The figure panels

In [ ]:
# (a) within-class d^2 / overall d^2: horizontal boxes
ORDER_A = ["pixels", "wect", "wect_pca", "wect_align", "diwect"]
LABELS_A = {"pixels": "Pixels", "wect": "WECT", "wect_pca": "WECT PCA", "wect_align": "WECT + align",
            "diwect": "DIWECT"}
fig, ax = plt.subplots(figsize=(3.0, 4.6))
ys = np.arange(len(ORDER_A))
samples = [df.loc[df.method == m, "within_class"].to_numpy() for m in ORDER_A]
C.box_plot(ax, samples, ys, [C.color(ROLE[m]) for m in ORDER_A], width=0.55, vert=False)
for y, v in zip(ys, samples):   # median beside each box, on the side away from the axis or the dashed line at 1
    near_one = v.max() > 0.5
    ax.annotate(f"{np.median(v):.2f}", (v.min() - 0.02 if near_one else v.max() + 0.02, y),
                ha="right" if near_one else "left", va="center", fontsize=7.5)
ax.axvline(1.0, ls="--", lw=1, color="k")
ax.annotate("class explains nothing", (0.985, -0.75), ha="right", va="center", fontsize=7.5)   # above the top row
ax.set_yticks(ys); ax.set_yticklabels([LABELS_A[m] for m in ORDER_A])
ax.set_ylim(len(ORDER_A) - 0.5, -1.1)   # inverted: first row on top, headroom for the label
ax.tick_params(axis="y", length=0)
ax.set_xlim(0, 1.05)
ax.set_xlabel("within-class $d^2$ / overall $d^2$")
C.save_pdf(fig, FOLDER, "figure3a-within-class")


# (b) MDS embeddings
LEGEND_STYLE = dict(frameon=True, framealpha=0.85, edgecolor="none", borderpad=0.3, handletextpad=0.2,
                    columnspacing=0.6, handlelength=1.0, borderaxespad=0.2)
MARKERS = ["o", "s", "^", "D", "P"]


def embed(ax, Z, colors, title, cmap, legend=None, markers=None, s=11, title_fs=9.5, legend_fs=6.5, ms=5):
    if markers is None:
        ax.scatter(Z[:, 0], Z[:, 1], c=colors, s=s, cmap=cmap, alpha=0.85, edgecolors="none")
    else:
        norm = plt.Normalize(colors.min(), colors.max())
        for k in np.unique(colors):
            sel = colors == k
            ax.scatter(Z[sel, 0], Z[sel, 1], color=plt.get_cmap(cmap)(norm(k)), s=13,
                       marker=markers[int(k) % len(markers)], alpha=0.85, edgecolors="none")
    lo, hi = np.percentile(Z, [1, 99], axis=0)
    pad = 0.12 * (hi - lo + 1e-9)
    ax.set_xlim(lo[0] - pad[0], hi[0] + pad[0]); ax.set_ylim(lo[1] - pad[1], hi[1] + pad[1])
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=title_fs)
    if legend:
        handles = [plt.Line2D([], [], marker=MARKERS[i % len(MARKERS)] if markers else "o", ls="", ms=ms,
                              color=plt.get_cmap(cmap)(i / max(len(legend) - 1, 1)))
                   for i in range(len(legend))]
        ax.legend(handles, legend, fontsize=legend_fs, ncol=2, loc="best", **LEGEND_STYLE)


ROT_COLORS = ["#0072B2", "#E69F00", "#6A3D9A", "#D55E00", "#CC79A7"]
ROT_LABELS = [f"{int(360 * k / K)}–{int(360 * (k + 1) / K)}°" for k in range(K)]


def embed_rotation(ax, Z, rot_bin, title, s=11, title_fs=9.5, legend_fs=6.5, ms=5):
    ax.scatter(Z[:, 0], Z[:, 1], c=[ROT_COLORS[b] for b in rot_bin], s=s, alpha=0.85, edgecolors="none")
    lo, hi = np.percentile(Z, [1, 99], axis=0)
    pad = 0.12 * (hi - lo + 1e-9)
    ax.set_xlim(lo[0] - pad[0], hi[0] + pad[0]); ax.set_ylim(lo[1] - pad[1], hi[1] + pad[1])
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=title_fs)
    if legend_fs:   # in-panel legend with one legend per column
        handles = [plt.Line2D([], [], marker="o", ls="", ms=ms, color=c) for c in ROT_COLORS]
        ax.legend(handles, ROT_LABELS, fontsize=legend_fs, ncol=2, loc="best", **LEGEND_STYLE)


fig, axes = plt.subplots(2, 2, figsize=(3.75, 4.1))
for row, rep in enumerate(("wect", "diwect")):
    embed(axes[row, 0], embeddings[rep], last["y"], f"{C.label(rep)}: shape", "tab10", s=2.5, title_fs=8.5)
    embed_rotation(axes[row, 1], embeddings[rep], last["pose"], f"{C.label(rep)}: rotation bin",
                   s=2.5, title_fs=8.5, legend_fs=None)
shape_cm = plt.get_cmap("tab10")
shape_handles = [plt.Line2D([], [], marker="o", ls="", ms=3.5, color=shape_cm(i / (len(shapes) - 1)))
                 for i in range(len(shapes))]
rot_handles = [plt.Line2D([], [], marker="o", ls="", ms=3.5, color=c) for c in ROT_COLORS]
for ax, (h, l) in zip(axes[1], [(shape_handles, list(shapes)), (rot_handles, ROT_LABELS)]):
    ax.legend(h, l, fontsize=7.5, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.06),
              frameon=False, borderpad=0.4,
              handletextpad=0.2, columnspacing=0.8, handlelength=1.0, borderaxespad=0)
C.save_pdf(fig, FOLDER, "figure3b-mds")
